# Models and Multilinguality

This notebook gathers task/corpus-level multilingual capacity measures that are separate from the LLID `run_eval.py` diagnostics.

Main table metric:

- **INCLUDE score**: macro-average accuracy over the 44 zero-shot/default INCLUDE language groups from upstream `lm_eval`.

Auxiliary diagnostics:

- **INCLUDE micro accuracy**: sample-count weighted accuracy over the same language groups.
- **BPB**: bits per UTF-8 byte on `pud21_ud6`.
- **PUD line logprob**: average summed line log-probability on PUD-only rows, using `source == "pud"` because UD6 is not parallel.

In [ ]:
from pathlib import Path
import json
import math
import re
import sys

import numpy as np
import pandas as pd

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "README.md").exists():
    REPO_ROOT = REPO_ROOT.parent

LM_HARNESS_ROOT = REPO_ROOT / "logs" / "lm_harness" / "include"
BPB_ROOT = REPO_ROOT / "logs" / "bpb"
EXPORT_DIR = REPO_ROOT / "logs" / "analysis_exports" / "model_multilinguality"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 180)

In [ ]:
MODEL_META = [
    {
        "slug": "gpt2",
        "display": "GPT-2",
        "latex": r"GPT-2 \citep{radford2019gpt2}",
    },
    {
        "slug": "gpt2xl",
        "display": "GPT-2 XL",
        "latex": r"GPT-2-XL",
    },
    {
        "slug": "llama27b",
        "display": "Llama-2-7B",
        "latex": r"Llama-2-7B \citep{touvron2023llama2}",
    },
    {
        "slug": "llama318b",
        "display": "Llama-3.1-8B",
        "latex": r"Llama-3.1-8B \citep{grattafiori2024llama3herdmodels}",
    },
    {
        "slug": "llama318binstr",
        "display": "Llama-3.1-8B-Instruct",
        "latex": r"Llama-3.1-8B-Instruct",
    },
    {
        "slug": "olmo211247b",
        "display": "OLMo-2-1124-7B",
        "latex": r"OLMo-2-1124-7B \citep{walsh2025olmo2}",
    },
    {
        "slug": "apertus8b",
        "display": "Apertus-8B",
        "latex": r"Apertus-8B \citep{apertus2025apertus}",
    },
    {
        "slug": "apertus8binstr",
        "display": "Apertus-8B-Instruct",
        "latex": r"Apertus-8B-Instruct",
    },
    {
        "slug": "nemo",
        "display": "Mistral-Nemo-Instruct-2407",
        "latex": r"Mistral-Nemo-Instruct-2407 \citep{mistralai2024mistralnemo}",
    },
    {
        "slug": "aya238b",
        "display": "Aya-23-8B",
        "latex": r"Aya-23-8B \citep{aryabumi2024aya23}",
    },
    {
        "slug": "eurollm9b",
        "display": "EuroLLM-9B",
        "latex": r"EuroLLM-9B \citep{martins2025eurollm}",
    },
    {
        "slug": "eurollm9binstr",
        "display": "EuroLLM-9B-Instruct",
        "latex": r"EuroLLM-9B-Instruct",
    },
]

MODEL_DF = pd.DataFrame(MODEL_META)
MODEL_ORDER = MODEL_DF["slug"].tolist()
MODEL_LABELS = MODEL_DF.set_index("slug")["display"].to_dict()
MODEL_LATEX = MODEL_DF.set_index("slug")["latex"].to_dict()
MODEL_RANK = {slug: i for i, slug in enumerate(MODEL_ORDER)}
MODEL_DF

In [ ]:
def latest_json(paths):
    paths = sorted(paths)
    if not paths:
        return None
    return max(paths, key=lambda p: p.stat().st_mtime_ns)


def load_include_setting(setting: str) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Return language-level and model-level INCLUDE rows for one lm-eval setting."""
    setting_root = LM_HARNESS_ROOT / setting
    language_rows = []
    model_rows = []
    if not setting_root.exists():
        return pd.DataFrame(), pd.DataFrame()

    for model_dir in sorted(p for p in setting_root.iterdir() if p.is_dir()):
        slug = model_dir.name
        result_path = latest_json(model_dir.glob("*/results_*.json"))
        if result_path is None:
            continue
        data = json.loads(result_path.read_text(encoding="utf-8"))
        groups = data.get("groups", {})
        if not groups:
            continue

        for group_name, metrics in groups.items():
            acc = metrics.get("acc,none")
            sample_count = metrics.get("sample_count", {}).get("acc,none", metrics.get("sample_len"))
            if acc is None:
                continue
            lang = group_name.removeprefix("include_base_44_")
            lang = lang.removesuffix("_few_shot_en").removesuffix("_few_shot_og")
            language_rows.append(
                {
                    "setting": setting,
                    "model_slug": slug,
                    "model_name": data.get("model_name"),
                    "lang": lang,
                    "acc": float(acc),
                    "sample_count": int(sample_count) if sample_count is not None else np.nan,
                    "result_path": result_path.name,
                    "result_mtime_ns": result_path.stat().st_mtime_ns,
                }
            )

        frame = pd.DataFrame([row for row in language_rows if row["setting"] == setting and row["model_slug"] == slug])
        if frame.empty:
            continue
        total_weight = frame["sample_count"].sum()
        micro_acc = np.average(frame["acc"], weights=frame["sample_count"]) if total_weight > 0 else np.nan
        model_rows.append(
            {
                "setting": setting,
                "model_slug": slug,
                "model_name": data.get("model_name"),
                "include_macro_acc": frame["acc"].mean(),
                "include_micro_acc": micro_acc,
                "include_n_langs": frame["lang"].nunique(),
                "include_n_examples": int(total_weight) if total_weight > 0 else np.nan,
                "result_path": result_path.name,
                "result_mtime_ns": result_path.stat().st_mtime_ns,
            }
        )

    return pd.DataFrame(language_rows), pd.DataFrame(model_rows)

include_language_frames = []
include_model_frames = []
for setting in ["include_base_44", "include_base_44_few_shot_en", "include_base_44_few_shot_og"]:
    lang_df, model_df = load_include_setting(setting)
    include_language_frames.append(lang_df)
    include_model_frames.append(model_df)

include_lang_df = pd.concat(include_language_frames, ignore_index=True) if include_language_frames else pd.DataFrame()
include_model_df = pd.concat(include_model_frames, ignore_index=True) if include_model_frames else pd.DataFrame()

if not include_model_df.empty:
    include_model_df["display_model"] = include_model_df["model_slug"].map(MODEL_LABELS).fillna(include_model_df["model_slug"])
    include_model_df["model_rank"] = include_model_df["model_slug"].map(MODEL_RANK).fillna(999).astype(int)
    include_model_df = include_model_df.sort_values(["setting", "model_rank", "model_slug"])

include_model_df

In [ ]:
# Zero-shot/default INCLUDE table source. This is the paper-table metric unless noted otherwise.
include_default_df = include_model_df[include_model_df["setting"].eq("include_base_44")].copy() if not include_model_df.empty else pd.DataFrame()
include_default_df[[
    "display_model", "model_slug", "include_macro_acc", "include_micro_acc", "include_n_langs", "include_n_examples", "result_path"
]] if not include_default_df.empty else include_default_df

In [ ]:
def load_bpb_summaries() -> pd.DataFrame:
    rows = []
    for path in sorted(BPB_ROOT.glob("*_pud21_ud6.csv")):
        if path.name.startswith("smoke_") or path.name.startswith("smoke"):
            continue
        frame = pd.read_csv(path)
        frame["summary_path"] = str(path.relative_to(REPO_ROOT))
        rows.append(frame)
    if not rows:
        return pd.DataFrame()
    out = pd.concat(rows, ignore_index=True)
    out["display_model"] = out["model_slug"].map(MODEL_LABELS).fillna(out["model_slug"])
    out["model_rank"] = out["model_slug"].map(MODEL_RANK).fillna(999).astype(int)
    return out.sort_values(["model_rank", "model_slug", "lang"])

bpb_df = load_bpb_summaries()
bpb_overall_df = bpb_df[bpb_df["lang"].eq("overall")].copy() if not bpb_df.empty else pd.DataFrame()
bpb_overall_df[[
    "display_model", "model_slug", "bits_per_byte", "nll_bits_per_char", "mean_token_nll_nats", "mean_line_logprob_bits", "n_examples", "summary_path"
]] if not bpb_overall_df.empty else bpb_overall_df

In [ ]:
def load_pud_line_summaries() -> pd.DataFrame:
    rows = []
    for path in sorted(BPB_ROOT.glob("*_pud21_ud6_lines.csv")):
        if path.name.startswith("smoke_") or path.name.startswith("smoke"):
            continue
        frame = pd.read_csv(path)
        if "source" not in frame.columns:
            continue
        frame = frame[frame["source"].eq("pud")].copy()
        if frame.empty:
            continue
        rows.append(
            {
                "model_slug": frame["model_slug"].iloc[0],
                "display_model": MODEL_LABELS.get(frame["model_slug"].iloc[0], frame["model_slug"].iloc[0]),
                "pud_mean_line_logprob_bits": frame["logprob_bits"].mean(),
                "pud_mean_line_nll_bits": -frame["logprob_bits"].mean(),
                "pud_median_line_nll_bits": -frame["logprob_bits"].median(),
                "pud_n_parallel_ids": frame["parallel_id"].nunique(),
                "pud_n_lines": len(frame),
                "line_path": str(path.relative_to(REPO_ROOT)),
            }
        )
    out = pd.DataFrame(rows)
    if out.empty:
        return out
    out["model_rank"] = out["model_slug"].map(MODEL_RANK).fillna(999).astype(int)
    return out.sort_values(["model_rank", "model_slug"])

pud_line_df = load_pud_line_summaries()
pud_line_df

In [ ]:
# Combined diagnostic overview.
overview = MODEL_DF[["slug", "display"]].rename(columns={"slug": "model_slug", "display": "display_model"}).copy()
if not include_default_df.empty:
    overview = overview.merge(
        include_default_df[["model_slug", "include_macro_acc", "include_micro_acc", "include_n_langs", "include_n_examples"]],
        on="model_slug",
        how="left",
    )
if not bpb_overall_df.empty:
    overview = overview.merge(
        bpb_overall_df[["model_slug", "bits_per_byte", "nll_bits_per_char", "mean_token_nll_nats"]],
        on="model_slug",
        how="left",
    )
if not pud_line_df.empty:
    overview = overview.merge(
        pud_line_df[["model_slug", "pud_mean_line_nll_bits", "pud_n_parallel_ids", "pud_n_lines"]],
        on="model_slug",
        how="left",
    )

overview.to_csv(EXPORT_DIR / "overview.csv", index=False)
overview

In [ ]:
# Optional: per-language INCLUDE scores for diagnostics.
if not include_lang_df.empty:
    include_lang_df["display_model"] = include_lang_df["model_slug"].map(MODEL_LABELS).fillna(include_lang_df["model_slug"])
    include_lang_df["model_rank"] = include_lang_df["model_slug"].map(MODEL_RANK).fillna(999).astype(int)
    include_lang_df.sort_values(["setting", "model_rank", "lang"]).to_csv(EXPORT_DIR / "include_language_scores.csv", index=False)

include_lang_df.head() if not include_lang_df.empty else include_lang_df

In [ ]:
def format_score(value, *, scale=1.0, digits=1):
    if pd.isna(value):
        return "--"
    return f"{value * scale:.{digits}f}"


def make_latex_table(table_df: pd.DataFrame) -> str:
    bs = chr(92)
    row_end = " " + bs + bs
    lines = []
    lines.append(bs + "begin{table*}[t]")
    lines.append(bs + "centering")
    lines.append(bs + "small")
    lines.append(bs + "begin{tabular}{lrrrrr}")
    lines.append(bs + "toprule")
    lines.append(bs + "textbf{Model} & " + bs + "textbf{INCLUDE 0-shot} & " + bs + "textbf{PUD+UD BPB} & " + bs + "textbf{PUD+UD bpc} & " + bs + "textbf{PUD+UD token NLL} & " + bs + "textbf{PUD line NLL}" + row_end)
    lines.append(bs + "midrule")
    for row in table_df.itertuples(index=False):
        model = MODEL_LATEX[row.model_slug]
        include_score = format_score(row.include_macro_acc, scale=100.0, digits=1)
        bpb = format_score(row.bits_per_byte, digits=2)
        bpc = format_score(row.nll_bits_per_char, digits=2)
        token_nll = format_score(row.mean_token_nll_nats, digits=2)
        line_nll = format_score(row.pud_mean_line_nll_bits, digits=1)
        lines.append(f"{model} & {include_score} & {bpb} & {bpc} & {token_nll} & {line_nll}" + row_end)
    lines.append(bs + "bottomrule")
    lines.append(bs + "end{tabular}")
    lines.append(bs + "caption{" + bs + "textbf{Model-level multilingual capacity diagnostics.} INCLUDE 0-shot is macro-average accuracy over the 44 upstream INCLUDE language groups, reported as a percentage. PUD+UD BPB is bits per UTF-8 byte on the combined PUD21+UD6 corpus; bpc is negative log-likelihood in bits per Unicode codepoint; token NLL is mean token negative log-likelihood in nats. PUD line NLL is the mean summed sentence negative log-likelihood in bits on PUD-only parallel rows. Lower is better for the PUD likelihood metrics.}")
    lines.append(bs + "label{tab:model-include-scores}")
    lines.append(bs + "end{table*}")
    return "\n".join(lines)

latex_table_df = MODEL_DF[["slug", "display"]].rename(columns={"slug": "model_slug"}).copy()
latex_table_df = latex_table_df.merge(
    include_default_df[["model_slug", "include_macro_acc"]] if not include_default_df.empty else pd.DataFrame(columns=["model_slug", "include_macro_acc"]),
    on="model_slug",
    how="left",
)
latex_table_df = latex_table_df.merge(
    bpb_overall_df[["model_slug", "bits_per_byte", "nll_bits_per_char", "mean_token_nll_nats"]] if not bpb_overall_df.empty else pd.DataFrame(columns=["model_slug", "bits_per_byte", "nll_bits_per_char", "mean_token_nll_nats"]),
    on="model_slug",
    how="left",
)
latex_table_df = latex_table_df.merge(
    pud_line_df[["model_slug", "pud_mean_line_nll_bits"]] if not pud_line_df.empty else pd.DataFrame(columns=["model_slug", "pud_mean_line_nll_bits"]),
    on="model_slug",
    how="left",
)
latex_table = make_latex_table(latex_table_df)
(EXPORT_DIR / "scores_table.tex").write_text(latex_table + "\n", encoding="utf-8")
print(latex_table)

In [ ]:
# Wide inspection table: all currently gathered multilinguality scores.
def make_all_scores_latex_table() -> str:
    bs = chr(92)
    row_end = " " + bs + bs

    include_wide = pd.DataFrame({"model_slug": MODEL_ORDER})
    if not include_model_df.empty:
        inc = include_model_df.pivot_table(
            index="model_slug",
            columns="setting",
            values=["include_macro_acc", "include_micro_acc"],
            aggfunc="first",
        )
        inc.columns = [f"{metric}_{setting}" for metric, setting in inc.columns]
        inc = inc.reset_index()
        include_wide = include_wide.merge(inc, on="model_slug", how="left")

    all_df = MODEL_DF[["slug", "display"]].rename(columns={"slug": "model_slug"}).copy()
    all_df = all_df.merge(include_wide, on="model_slug", how="left")
    all_df = all_df.merge(
        bpb_overall_df[[
            "model_slug",
            "bits_per_byte",
            "nll_bits_per_char",
            "mean_token_nll_nats",
            "mean_line_nll_nats",
            "mean_line_logprob_bits",
        ]] if not bpb_overall_df.empty else pd.DataFrame(columns=[
            "model_slug",
            "bits_per_byte",
            "nll_bits_per_char",
            "mean_token_nll_nats",
            "mean_line_nll_nats",
            "mean_line_logprob_bits",
        ]),
        on="model_slug",
        how="left",
    )
    all_df = all_df.merge(
        pud_line_df[[
            "model_slug",
            "pud_mean_line_logprob_bits",
            "pud_mean_line_nll_bits",
            "pud_median_line_nll_bits",
            "pud_n_parallel_ids",
            "pud_n_lines",
        ]] if not pud_line_df.empty else pd.DataFrame(columns=[
            "model_slug",
            "pud_mean_line_logprob_bits",
            "pud_mean_line_nll_bits",
            "pud_median_line_nll_bits",
            "pud_n_parallel_ids",
            "pud_n_lines",
        ]),
        on="model_slug",
        how="left",
    )

    def pct(col, row):
        return format_score(getattr(row, col, np.nan), scale=100.0, digits=1)

    def num(col, row, digits=2):
        return format_score(getattr(row, col, np.nan), digits=digits)

    columns = [
        "Model",
        "0 macro",
        "0 micro",
        "5en macro",
        "5en micro",
        "5og macro",
        "5og micro",
        "BPB",
        "bpc",
        "tok NLL",
        "PUD+UD line logp",
        "PUD+UD line NLL",
        "PUD line logp",
        "PUD line NLL",
        "PUD med line NLL",
    ]

    lines = []
    lines.append(bs + "begin{table*}[t]")
    lines.append(bs + "centering")
    lines.append(bs + "scriptsize")
    lines.append(bs + "resizebox{" + bs + "textwidth}{!}{%")
    lines.append(bs + "begin{tabular}{lrrrrrrrrrrrrrr}")
    lines.append(bs + "toprule")
    lines.append(" & ".join(bs + f"textbf{{{c}}}" if c == "Model" else bs + f"textbf{{{c}}}" for c in columns) + row_end)
    lines.append(bs + "midrule")
    for row in all_df.itertuples(index=False):
        values = [
            MODEL_LATEX[row.model_slug],
            pct("include_macro_acc_include_base_44", row),
            pct("include_micro_acc_include_base_44", row),
            pct("include_macro_acc_include_base_44_few_shot_en", row),
            pct("include_micro_acc_include_base_44_few_shot_en", row),
            pct("include_macro_acc_include_base_44_few_shot_og", row),
            pct("include_micro_acc_include_base_44_few_shot_og", row),
            num("bits_per_byte", row, 2),
            num("nll_bits_per_char", row, 2),
            num("mean_token_nll_nats", row, 2),
            num("mean_line_logprob_bits", row, 1),
            num("mean_line_nll_nats", row, 1),
            num("pud_mean_line_logprob_bits", row, 1),
            num("pud_mean_line_nll_bits", row, 1),
            num("pud_median_line_nll_bits", row, 1),
        ]
        lines.append(" & ".join(values) + row_end)
    lines.append(bs + "bottomrule")
    lines.append(bs + "end{tabular}%")
    lines.append("}")
    lines.append(bs + "caption{" + bs + "textbf{All gathered model-level multilinguality diagnostics.} INCLUDE columns are macro- and micro-average accuracies in percent for zero-shot/default (0), English 5-shot (5en), and original-language 5-shot (5og), when available. BPB is bits per UTF-8 byte on PUD21+UD6; bpc is bits per Unicode codepoint; tok NLL is mean token negative log-likelihood in nats; PUD+UD line logp is mean line log-probability in bits and PUD+UD line NLL is the corresponding mean negative log-likelihood in nats over the full PUD21+UD6 corpus; PUD line logp/NLL columns use only PUD parallel rows and are in bits.}")
    lines.append(bs + "label{tab:model-multilinguality-all-scores}")
    lines.append(bs + "end{table*}")
    return "\n".join(lines), all_df

all_scores_latex, all_scores_df = make_all_scores_latex_table()
all_scores_df.to_csv(EXPORT_DIR / "all_scores.csv", index=False)

all_scores_display_df = pd.DataFrame({
    "Model": [MODEL_LATEX[slug] for slug in all_scores_df["model_slug"]],
    "0 macro": all_scores_df["include_macro_acc_include_base_44"].map(lambda value: format_score(value, scale=100.0, digits=1)),
    "0 micro": all_scores_df["include_micro_acc_include_base_44"].map(lambda value: format_score(value, scale=100.0, digits=1)),
    "5en macro": all_scores_df.get("include_macro_acc_include_base_44_few_shot_en", pd.Series(np.nan, index=all_scores_df.index)).map(lambda value: format_score(value, scale=100.0, digits=1)),
    "5en micro": all_scores_df.get("include_micro_acc_include_base_44_few_shot_en", pd.Series(np.nan, index=all_scores_df.index)).map(lambda value: format_score(value, scale=100.0, digits=1)),
    "5og macro": all_scores_df.get("include_macro_acc_include_base_44_few_shot_og", pd.Series(np.nan, index=all_scores_df.index)).map(lambda value: format_score(value, scale=100.0, digits=1)),
    "5og micro": all_scores_df.get("include_micro_acc_include_base_44_few_shot_og", pd.Series(np.nan, index=all_scores_df.index)).map(lambda value: format_score(value, scale=100.0, digits=1)),
    "BPB": all_scores_df["bits_per_byte"].map(lambda value: format_score(value, digits=2)),
    "bpc": all_scores_df["nll_bits_per_char"].map(lambda value: format_score(value, digits=2)),
    "tok NLL": all_scores_df["mean_token_nll_nats"].map(lambda value: format_score(value, digits=2)),
    "PUD+UD line logp": all_scores_df["mean_line_logprob_bits"].map(lambda value: format_score(value, digits=1)),
    "PUD+UD line NLL": all_scores_df["mean_line_nll_nats"].map(lambda value: format_score(value, digits=1)),
    "PUD line logp": all_scores_df["pud_mean_line_logprob_bits"].map(lambda value: format_score(value, digits=1)),
    "PUD line NLL": all_scores_df["pud_mean_line_nll_bits"].map(lambda value: format_score(value, digits=1)),
    "PUD med line NLL": all_scores_df["pud_median_line_nll_bits"].map(lambda value: format_score(value, digits=1)),
})
all_scores_display_df.to_csv(EXPORT_DIR / "all_scores_display.csv", index=False)

(EXPORT_DIR / "all_scores_table.tex").write_text(all_scores_latex + "\n", encoding="utf-8")
print(all_scores_latex)

In [ ]:
# Headline paper table: primary INCLUDE score plus likelihood checks.
def make_headline_multilinguality_table() -> tuple[str, pd.DataFrame]:
    bs = chr(92)
    row_end = " " + bs + bs
    metric_cols = [
        "model_slug",
        "include_macro_acc_include_base_44",
        "pud_mean_line_nll_bits",
        "bits_per_byte",
    ]
    headline_df = all_scores_df[metric_cols].copy()
    headline_df = headline_df.sort_values("include_macro_acc_include_base_44", ascending=False, na_position="last")

    def rank_values(series: pd.Series, *, ascending: bool) -> pd.Series:
        return series.rank(method="first", ascending=ascending, na_option="bottom").astype(int)

    headline_df["rank_include"] = rank_values(headline_df["include_macro_acc_include_base_44"], ascending=False)
    headline_df["rank_pud_nll"] = rank_values(headline_df["pud_mean_line_nll_bits"], ascending=True)
    headline_df["rank_bpb"] = rank_values(headline_df["bits_per_byte"], ascending=True)

    def highlight_cell(text: str, rank: int) -> str:
        if text == "--":
            return text
        if rank == 1:
            return bs + "textbf{" + text + "}"
        if rank == 2:
            return bs + "underline{" + text + "}"
        return text

    display_df = pd.DataFrame({
        "Model": [MODEL_LATEX[slug] for slug in headline_df["model_slug"]],
        "INCLUDE 0 macro": [
            highlight_cell(format_score(row.include_macro_acc_include_base_44, scale=100.0, digits=1), row.rank_include)
            for row in headline_df.itertuples(index=False)
        ],
        "PUD line NLL": [
            highlight_cell(format_score(row.pud_mean_line_nll_bits, digits=1), row.rank_pud_nll)
            for row in headline_df.itertuples(index=False)
        ],
        "BPB": [
            highlight_cell(format_score(row.bits_per_byte, digits=2), row.rank_bpb)
            for row in headline_df.itertuples(index=False)
        ],
    })

    lines = []
    lines.append(bs + "begin{table}")
    lines.append(bs + "centering")
    lines.append(bs + "resizebox{1.0" + bs + "linewidth}{!}{")
    lines.append("    " + bs + "begin{tabular}{lccc}")
    lines.append("        " + bs + "toprule")
    lines.append("        " + bs + "textbf{Model} &")
    lines.append("        " + bs + "multicolumn{1}{c}{" + bs + "makecell{" + bs + "textbf{INCLUDE}" + bs + bs + bs + "textbf{0-shot macro} ($" + bs + "uparrow$)}} &")
    lines.append("        " + bs + "multicolumn{1}{c}{" + bs + "makecell{" + bs + "textbf{PUD21 line}" + bs + bs + bs + "textbf{NLL} ($" + bs + "downarrow$)}} &")
    lines.append("        " + bs + "multicolumn{1}{c}{" + bs + "makecell{" + bs + "textbf{PUD21+UD6}" + bs + bs + bs + "textbf{BPB} ($" + bs + "downarrow$)}} " + row_end)
    lines.append("        " + bs + "midrule")
    for _, row in display_df.iterrows():
        lines.append(
            f"        {row['Model']} & {row['INCLUDE 0 macro']} & {row['PUD line NLL']} & {row['BPB']}" + row_end
        )
    lines.append("        " + bs + "bottomrule")
    lines.append("    " + bs + "end{tabular}")
    lines.append("}")
    lines.append(bs + "caption{" + bs + "textbf{Multilingual Performance.} Models are sorted by zero-shot macro accuracy on INCLUDE, our primary downstream QA task. Line NLL is an auxiliary likelihood score computed on a parallel corpus across languages. BPB is likelihood normalized by UTF-8 byte count, but it remains script-sensitive because byte counts differ across writing systems. Best is bold; second-best is underlined.}")
    lines.append(bs + "label{tab:model-stats}")
    lines.append(bs + "end{table}")
    return "\n".join(lines), display_df

headline_latex, headline_display_df = make_headline_multilinguality_table()
headline_display_df.to_csv(EXPORT_DIR / "headline_scores_display.csv", index=False)
headline_latex_path = EXPORT_DIR / "headline_table.tex"
headline_latex_path.write_text(headline_latex + "\n", encoding="utf-8")
print(headline_latex)


In [ ]:
# Completion check: models without a zero-shot/default INCLUDE result yet.
missing_include = latex_table_df[latex_table_df["include_macro_acc"].isna()][["model_slug", "display"]]
missing_include
